# Laboratorio Fase 1 & 2 CRISP-DM: Enfoque Baseline desde Cero

## 1. Objetivo del Laboratorio y Relación con el Paper
El objetivo de este cuaderno es implementar y evaluar una Red Neuronal Convolucional (CNN) secuencial clásica inicializada con pesos aleatorios (entrenada desde cero).

**Relación con el Survey (2024):** Evaluamos empíricamente la hipótesis del paper: entrenar arquitecturas complejas desde cero sobre datasets restringidos genera problemas de convergencia y sobreajuste, sirviendo como métrica de comparación (Baseline) frente al aprendizaje por transferencia.

In [ ]:
# Limpiamos intentos anteriores por si acaso
!rm -rf src_folder

# Clonación del código fuente (src/)
!git clone https://github.com/Esme0123/CasoEstudio3.git src_folder
!cp -r src_folder/src .

# Instalamos SOLO lo que falta en Colab (timm y grad-cam).
# torch, numpy, etc. se usan en la versión que ya trae Colab -> sin conflictos ni reinicios.
!pip install -q -r src_folder/requirements.txt

print("Entorno listo. Continúa con la siguiente celda.")

In [ ]:
# Reproducibilidad: fijamos todas las semillas (random, numpy, torch, cuda)
from src.data_processing import set_seed

set_seed(42)
print("Semillas fijadas (SEED=42): resultados reproducibles entre ejecuciones.")

## 2. Ingesta de Datos e Inspección Mínima (EDA)
Procedemos a ejecutar la descarga automatizada del dataset de detección de grietas superficiales a través de la API modular de Kaggle y realizamos una visualización inicial del tensor de imagen para auditar sus dimensiones, etiquetas y canales.

In [ ]:
# Ingesta y EDA Minimo
from src.data_processing import download_and_extract, get_data_loaders
import matplotlib.pyplot as plt
import numpy as np

download_and_extract()
train_loader, test_loader = get_data_loaders()

# Inspección Visual (EDA)
images, labels = next(iter(train_loader))

# Las imágenes vienen normalizadas (mean/std de ImageNet). Las des-normalizamos
# antes de graficar para que los colores se vean correctamente y matplotlib no recorte valores.
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])
img = images[0].permute(1, 2, 0).numpy()
img = np.clip(std * img + mean, 0, 1)

plt.imshow(img)
plt.title(f"Clase Muestra: {labels[0].item()}")
plt.axis('off')
plt.show()

## 3. Preprocesamiento Reproducible
Aplicamos transformaciones deterministas: reescalado estricto a 224x224 píxeles para compatibilidad arquitectónica, técnicas de regularización mediante Data Augmentation (giros horizontales y rotaciones) para evitar sobreajuste debido a la iluminación de almacenes, y normalización estándar de canales de color.

In [ ]:
from src.data_processing import get_data_loaders

train_loader, test_loader = get_data_loaders()

## 4. Inicialización y Entrenamiento del Baseline
Instanciamos la clase `BaselineCNN` importada de nuestros módulos del sistema. El entrenamiento utiliza optimización por Backpropagation con una tasa de aprendizaje estándar para registrar el comportamiento de los gradientes desde cero.

In [ ]:
#Entrenamiento guardando historial de métricas
import torch.optim as optim
import torch.nn as nn
import torch
from src.architecture_models import BaselineCNN
from src.data_processing import set_seed

# Semilla justo antes de instanciar el modelo: inicialización de pesos reproducible
set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BaselineCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Listas para almacenar la evolución por época
history = {'train_loss': [], 'train_acc': []}

print("Iniciando entrenamiento del Baseline (Desde Cero)...")
for epoch in range(5):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, lbls)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        _, predicted = outputs.max(1)
        total += lbls.size(0)
        correct += predicted.eq(lbls).sum().item()

    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = (correct / total) * 100
    history['train_loss'].append(epoch_loss)
    history['train_acc'].append(epoch_acc)
    print(f"Época {epoch+1}/5 -> Pérdida: {epoch_loss:.4f} | Precisión: {epoch_acc:.2f}%")

## 5. Métricas, Análisis de Error y Conclusiones

In [ ]:
#Generación de Gráficas, Reporte de Clasificación y Matriz de Confusión
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# 1. Renderizado de las curvas de aprendizaje
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Pérdida en Entrenamiento', color='#dc2626', linewidth=2)
plt.title('Evolución de la Función de Pérdida (Loss)')
plt.xlabel('Épocas')
plt.ylabel('Loss')
plt.grid(True, linestyle='--')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='Precisión en Entrenamiento', color='#2563eb', linewidth=2)
plt.title('Evolución de la Precisión (Accuracy)')
plt.xlabel('Épocas')
plt.ylabel('Porcentaje (%)')
plt.grid(True, linestyle='--')
plt.legend()
plt.tight_layout()
plt.show()

# 2. Extracción de predicciones sobre el conjunto de prueba (Test)
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, lbls in test_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(lbls.numpy())

# 3. Impresión del Reporte Formal de Métricas (F1-Score)
print("\n" + "="*50)
print("     REPORTE MÍNIMO UNIFORME DE CLASIFICACIÓN (BASELINE)")
print("="*50)
print(classification_report(all_labels, all_preds, target_names=['Perfecto (Negative)', 'Dañado (Positive)']))

# 4. Construcción visual de la Matriz de Confusión
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', xticklabels=['Perfecto', 'Dañado'], yticklabels=['Perfecto', 'Dañado'])
plt.title('Matriz de Confusión - Red desde Cero')
plt.ylabel('Clase Real (True Labels)')
plt.xlabel('Predicción del Modelo (Predicted Labels)')
plt.show()

### Interpretación Científica de Resultados (Fase de Evaluación - CRISP-DM)

1. **Análisis de las Curvas de Aprendizaje:** Contrario a lo que predeciría la hipótesis del paper para datasets restringidos, la CNN entrenada desde cero converge de forma estable y alcanza una precisión de entrenamiento muy alta (≈99.5% en 5 épocas). Esto se explica por la naturaleza del dataset *Surface Crack Detection*: es un conjunto grande (~40.000 imágenes) y visualmente muy separable (grietas marcadas frente a superficies lisas), por lo que incluso filtros convolucionales inicializados aleatoriamente aprenden rápidamente los bordes y discontinuidades que distinguen ambas clases.
2. **Evaluación del F1-Score:** El reporte de clasificación muestra un F1-Score alto en el conjunto de prueba. Por tanto, en este problema concreto el baseline **no** es inutilizable; el límite del enfoque "desde cero" no aparece en la métrica final sino en el costo: entrena la totalidad de los parámetros sin conocimiento previo, lo que aumenta el riesgo de sobreajuste y la dependencia de grandes volúmenes de datos etiquetados.
3. **Diagnóstico de la Matriz de Confusión:** La matriz de confusión confirma una baja tasa de error en ambas clases. Los pocos *Falsos Negativos* que aparezcan (empaques dañados clasificados como perfectos) siguen siendo el error crítico de negocio para la sucursal de La Paz, ya que implican que un producto defectuoso llegue al consumidor; por ello se vigilan específicamente en la comparación con MobileNetV4.

**Matiz frente a la hipótesis del paper:** La inviabilidad de entrenar desde cero que describe el survey se manifiesta sobre todo en datasets pequeños y con clases sutiles. En este dataset grande y de alto contraste el baseline rinde bien, así que la ventaja del *Transfer Learning* (Notebook 02) debe medirse en **velocidad de convergencia, número de parámetros entrenables y eficiencia para dispositivos edge**, más que en una diferencia drástica de precisión.